In [0]:
!pip install emoji nltk xgboost category_encoders catboost lightgbm optuna-integration[lightgbm]  

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Import Data

In [0]:
import pandas as pd
train_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/train.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)


In [0]:
import pandas as pd
test_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/test.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)

In [0]:
# effectiveness_map = {
#     "Highly Effective": 4,
#     "Considerably Effective": 3,
#     "Moderately Effective": 2,

#     "Marginally Effective": 1,
#         "Ineffective": 0,
# }
# train_df['effectiveness'] = train_df['effectiveness'].map(effectiveness_map)
# test_df['effectiveness'] = test_df['effectiveness'].map(effectiveness_map)

In [0]:
additional_stopwords = {
    "drug", "medicine", "tablet", "doctor", "patient", "review_benefits", "review_sideEffects",
    "review_overall", "mg", "day", "take", "took", "used", "taking", "pill", "one", "review",
    "dose", "dosage", "medication", "treatment", "therapy", "prescribed", "prescription", 
    "prescribe", "physician", "med", "antibiotic", "cream", "application", "apply", "use", 
    "using", "take", "taken", "stop", "stopped", "started", "starting", "start", "continue",
    "continued", "course", "treat", "treated", "treating", "dos", "mcg", "tab", "daily", 
    "nightly", "bedtime", "morning", "evening", "hour", "per", "pm", "qd", "weekly", "month",
    "week", "year", "night", "time", "two", "three", "four", "five", "every", "twice", "long",
    "within", "since", "around", "last", "next", "ago", "short",
    "january", "february", "march", "april", "may", "june", "july", "august", "september",
    "october", "november", "december", "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep",
    "sept", "oct", "nov", "dec"
}



In [0]:
# ==========================
# 1. Import Required Libraries
# ==========================
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from lightgbm import LGBMClassifier
import mlflow
from sklearn.impute import SimpleImputer
from category_encoders import CatBoostEncoder, TargetEncoder
import mlflow.sklearn
from sklearn.model_selection import cross_validate, StratifiedKFold
import optuna
import warnings
import emoji
warnings.filterwarnings("ignore")

nltk.download("stopwords")
nltk.download("wordnet")
low_cardinal_categorical_features=["side_effects"]
high_cardinal_categorical_features = ["medicine_name", "illness"]
numeric_features = ["rating"]
text_feature = ['review_benefits','review_sideEffects','review_overall']
import nltk

# ----------------------------------------------------------------
# 🔹 2. Stopwords and Lemmatizer setup
# ----------------------------------------------------------------

stop_words = set(stopwords.words('english')).union(additional_stopwords)
lemmatizer = WordNetLemmatizer()

# ----------------------------------------------------------------
# 🔹 3. Text Preprocessor Function
# ----------------------------------------------------------------
def preprocess_text_fast(text):
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, language='en')
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

def nlp_text(X):
    X = X.copy()
    for col in X.columns:
        X[col] = X[col].apply(preprocess_text_fast)
    return X

# Updated cleaner to return Series
def nlp_text_series(x):
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]  # Take first (only) column
    return x.apply(preprocess_text_fast)

# ----------------------------------------------------------------
# 🔹 4. Feature Groups
# ----------------------------------------------------------------

# ----------------------------------------------------------------
# 🔹 5. OneHotEncoder Pipeline (low-cardinality)
# ----------------------------------------------------------------
low_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
     ("scaler", StandardScaler())
])
CatBoostEncoder._get_tags = lambda self: {"allow_nan": True}
# ----------------------------------------------------------------
# 🔹 6. CatBoostEncoder Pipeline (high-cardinality)
# ----------------------------------------------------------------
high_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(fill_value="NOT_AVAILABLE", strategy="constant")),
    ("encoder", CatBoostEncoder()),
    ("scaler", StandardScaler())
])

# ----------------------------------------------------------------
# 🔹 7. NLP Transformer (text column)
# ----------------------------------------------------------------
nlp_pipeline = Pipeline([
    ("cleaner", FunctionTransformer(nlp_text_series, validate=False)),
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2)))
])

# ----------------------------------------------------------------
# 🔹 7. Numerical Transformer (Numerical column)
# ----------------------------------------------------------------
numeric_transformer = StandardScaler()

# Combine them
preprocessor = ColumnTransformer(
    transformers=[
        ("text", nlp_pipeline, text_feature),
        ("high_cat", high_categorical_transformer, high_cardinal_categorical_features),
        ("low_cat", low_categorical_transformer, low_cardinal_categorical_features),
        ("num", numeric_transformer, numeric_features),
    ]
)

X_train = train_df[["medicine_name", "rating", "side_effects", "illness", 'review_benefits','review_sideEffects','review_overall']]
y_train = train_df["effectiveness"]

X_test = test_df[["medicine_name", "rating", "side_effects", "illness", 'review_benefits','review_sideEffects','review_overall']]
y_test = test_df["effectiveness"]

# ==========================
# 7. Model Definition
# ==========================
# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))


# Define the objective function for Optuna
def objective(trial):
    param = {
        "objective": "multiclass",
        "num_class": 5,
        "metric": "multi_logloss",  # or "multi_error"
        "boosting_type": "gbdt",
        "random_state": 42,

        # --- Regularization ---
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 50.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 50.0, log=True),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 5.0),

        # --- Tree structure ---
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 10, 40),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 300),
        "max_bin": trial.suggest_int("max_bin", 100, 255),

        # --- Learning parameters ---
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        
        # --- Sampling ---
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),

        # --- Path smoothing ---
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 10.0),

        # --- Class balancing ---
        "class_weight": class_weight_dict,
    }

    model = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LGBMClassifier(**param)),
        ]
    )

    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_weighted",
        return_train_score=True,
        n_jobs=-1,
    )
    return cv_results["test_score"].mean()

# Create a study and optimize the objective function
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Get the best parameters
best_params = study.best_params
best_params["random_state"] = 537672287
best_params["class_weight"] = class_weight_dict
best_params["objective"] = "multiclass"
best_params["metric"]= "multi_logloss"
best_params["num_class"] = 5
# ==========================
# 9. Train & Evaluate
# ==========================


model_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(**best_params)),
    ]
)
print(f"Best params are: {best_params}")

# Fit the model
model_pipeline.fit(X_train, y_train)

# Stratified K-Fold for handling imbalanced classes
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=427)

# Evaluate model with cross-validation
cv_results = cross_validate(
    model_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=["f1_weighted"],
    return_train_score=True,
    n_jobs=-1
)
preds = model_pipeline.predict(X_test)

# Predict
preds = model_pipeline.predict(X_test)
print("=== Classification Report (Test) ===")
test_report_dict = classification_report(y_test, preds, output_dict=True)
print(classification_report(y_test, preds))
print("\n=== Confusion Matrix (Test) ===")
print(confusion_matrix(y_test, preds))
from sklearn.metrics import f1_score
print("\n=== Weighted F1 Score (Test) ===")
pd.DataFrame(test_report_dict).transpose().to_csv("classification_report_test.csv", index=True)

train_preds = model_pipeline.predict(X_train)
train_report_dict = classification_report(y_train, train_preds, output_dict=True)
print("\n=== Classification Report (Train) ===")
print(classification_report(y_train, train_preds))
print("\n=== Confusion Matrix (Train) ===")
print(confusion_matrix(y_train, train_preds))
print("\n=== Weighted F1 Score (Train) ===")
pd.DataFrame(train_report_dict).transpose().to_csv("classification_report_train.csv", index=True)



[nltk_data] Downloading package stopwords to /home/spark-65ea1bbf-
[nltk_data]     cf6e-4e7b-9c60-b1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/spark-65ea1bbf-
[nltk_data]     cf6e-4e7b-9c60-b1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[I 2025-11-06 07:11:14,409] A new study created in memory with name: no-name-e716b6d1-664d-435d-8fa7-3f5e5f4ad34d


[LightGBM] [Warning] min_gain_to_split is set=2.1533749462204947, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=2.1533749462204947
[LightGBM] [Warning] lambda_l2 is set=0.06224253199133724, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.06224253199133724
[LightGBM] [Warning] lambda_l1 is set=0.00016340097002557496, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00016340097002557496
[LightGBM] [Warning] min_gain_to_split is set=2.1533749462204947, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=2.1533749462204947
[LightGBM] [Warning] lambda_l2 is set=0.06224253199133724, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.06224253199133724
[LightGBM] [Warning] lambda_l1 is set=0.00016340097002557496, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00016340097002557496
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000192 seconds.
You can set `force_col_wise=true` 

/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-11-06 07:11:23,825] Trial 0 finished with value: 0.5923992178297106 and parameters: {'lambda_l1': 0.00016340097002557496, 'lambda_l2': 0.06224253199133724, 'min_gain_to_split': 2.1533749462204947, 'ma

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits wit

/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-11-06 07:11:27,670] Trial 1 finished with value: 0.5896041810850277 and parameters: {'lambda_l1': 0.0012389397400593782, 'lambda_l2': 0.0011704372617216926, 'min_gain_to_split': 2.598707615988132, 'ma

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits wit

/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-11-06 07:13:46,609] Trial 49 finished with value: 0.5882925909940726 and parameters: {'lambda_l1': 7.684047105416457, 'lambda_l2': 0.1688818652027415, 'min_gain_to_split': 3.32079094665761, 'max_depth

Best params are: {'lambda_l1': 34.9583758165214, 'lambda_l2': 0.02684447059566986, 'min_gain_to_split': 3.87280120932458, 'max_depth': 10, 'num_leaves': 17, 'min_child_samples': 74, 'max_bin': 153, 'learning_rate': 0.06469312262516418, 'n_estimators': 425, 'subsample': 0.8661229415063442, 'colsample_bytree': 0.7805348645553658, 'path_smooth': 6.275963280430011, 'random_state': 537672287, 'class_weight': {np.int64(0): np.float64(2.5157894736842104), np.int64(1): np.float64(3.322994652406417), np.int64(2): np.float64(1.4973493975903613), np.int64(3): np.float64(0.6696120689655173), np.int64(4): np.float64(0.467218045112782)}, 'objective': 'multiclass', 'metric': 'multi_logloss', 'num_class': 5}
[LightGBM] [Warning] min_gain_to_split is set=3.87280120932458, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=3.87280120932458
[LightGBM] [Warning] lambda_l2 is set=0.02684447059566986, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.02684447059566986
[LightGBM] [

/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_gain_to_split is set=3.87280120932458, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=3.87280120932458
[LightGBM] [Warning] lambda_l2 is set=0.02684447059566986, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.02684447059566986
[LightGBM] [Warning] lambda_l1 is set=34.9583758165214, reg_alpha=0.0 will be ignored. Current value: lambda_l1=34.9583758165214
[LightGBM] [Warning] min_gain_to_split is set=3.87280120932458, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=3.87280120932458
[LightGBM] [Warning] lambda_l2 is set=0.02684447059566986, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.02684447059566986
[LightGBM] [Warning] lambda_l1 is set=34.9583758165214, reg_alpha=0.0 will be ignored. Current value: lambda_l1=34.9583758165214
=== Classification Report (Test) ===
              precision    recall  f1-score   support

           0       0.49      0.88      0.63        82
           1       0.31 

In [0]:
# ==========================
# 1. Import Required Libraries
# ==========================
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
# from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from tensorflow.keras.optimizers import Adam, AdamW
import mlflow
import mlflow.keras
import warnings
warnings.filterwarnings("ignore")

nltk.download("stopwords")
nltk.download("wordnet")

# ==========================
# 2. Load Data
# ==========================
# train_df = pd.read_csv("train.tsv", sep="\t")
# test_df = pd.read_csv("test.tsv", sep="\t")

# ==========================
# 3. Basic Preprocessing
# ==========================
train_df["combined_review"] = (
    train_df["review_benefits"].astype(str) + " " +
    train_df["review_sideEffects"].astype(str) + " " +
    train_df["review_overall"].astype(str)
)

test_df["combined_review"] = (
    test_df["review_benefits"].astype(str) + " " +
    test_df["review_sideEffects"].astype(str) + " " +
    test_df["review_overall"].astype(str)
)

# ==========================
# 4. Clean Text Function
# ==========================
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(tokens)

train_df["combined_review"] = train_df["combined_review"].apply(clean_text)
test_df["combined_review"] = test_df["combined_review"].apply(clean_text)

# ==========================
# 5. Feature / Target Split
# ==========================
X_train = train_df[["medicine_name", "rating", "side_effects", "illness", "combined_review"]]
y_train = train_df["effectiveness"]

X_test = test_df[["medicine_name", "rating", "side_effects", "illness", "combined_review"]]
y_test = test_df["effectiveness"]

# Convert target to zero-based classes if categorical (Keras requires this)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
n_classes = len(le.classes_)

# ==========================
# 6. Preprocessing Pipeline
# ==========================
categorical_features = ["medicine_name", "illness", "side_effects"]
numeric_features = ["rating"]
text_feature = "combined_review"

categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numeric_transformer = StandardScaler()
text_transformer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, text_feature),
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features),
    ]
)

# ==========================
# 7. Define Neural Network
# ==========================

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# Compute class weights
# class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
# class_weight_dict = dict(enumerate(class_weights))
class_weight_dict = {
    0: 0.2,
    1: 0.5,
    2: 3.0,
    3: 4.0,
    4: 2.0
}

def create_nn_model(input_dim, num_classes):
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=l2(0.001), kernel_initializer='he_normal'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=AdamW(learning_rate=0.01),
        metrics=['accuracy'],
    )
    return model

# Custom wrapper to delay model building until input_dim known
class KerasNNWrapper:
    def __init__(self, num_classes, epochs=10, batch_size=64):
        self.num_classes = num_classes
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = None

    def fit(self, X, y):
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
        input_dim = X.shape[1]
        self.model = create_nn_model(input_dim, self.num_classes)
        self.model.fit(
            X, y,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=1,
            callbacks=[early_stop],
            validation_split=0.1,
            class_weight=class_weight_dict
        )
        return self

    def predict(self, X):
        preds = self.model.predict(X)
        return np.argmax(preds, axis=1)

# ==========================
# 8. Full Pipeline
# ==========================
nn_clf = KerasNNWrapper(num_classes=n_classes, epochs=30, batch_size=64)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", nn_clf)
    ]
)

# ==========================
# 9. Train & Evaluate
# ==========================
# Fit pipeline
model_pipeline.fit(X_train, y_train)

# Predict
preds = model_pipeline.predict(X_test)
print("=== Classification Report (Test) ===")
print(classification_report(y_test, preds))
print("\n=== Confusion Matrix (Test) ===")
print(confusion_matrix(y_test, preds))

train_preds = model_pipeline.predict(X_train)
print("\n=== Classification Report (Train) ===")
print(classification_report(y_train, train_preds))
print("\n=== Confusion Matrix (Train) ===")
print(confusion_matrix(y_train, train_preds))



[nltk_data] Downloading package stopwords to /home/spark-65ea1bbf-
[nltk_data]     cf6e-4e7b-9c60-b1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/spark-65ea1bbf-
[nltk_data]     cf6e-4e7b-9c60-b1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 9s 65ms/step - accuracy: 0.3863 - loss: 3.2775 - val_accuracy: 0.4437 - val_loss: 3.5795
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.4918 - loss: 3.2000 - val_accuracy: 0.4469 - val_loss: 3.0027
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5469 - loss: 2.4262 - val_accuracy: 0.3666 - val_loss: 2.7668
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.5744 - loss: 2.1827 - val_accuracy: 0.4630 - val_loss: 2.7337
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.6023 - loss: 2.0913 - val_accuracy: 0.4566 - val_loss: 2.8123
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6130 - loss: 2.0317 - val_accuracy: 0.5434 - val_loss: 2.4866
Epoch 7/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6227 - loss: 1.9323 - val_accuracy: 0.5113 - val_loss: 2.5958
Epoch 8/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.6230 - loss: 1.8570 - val_accuracy: 0.5338 - v